In [20]:
import torch
import time
import pandas as pd
from models_multi_head import PressureNetMultiHead
from datasets import HDF5Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as T

# === Configuration: adjust these as needed ===
BATCH_SIZES = [512, 1024, 2048, 4096]
WORKER_COUNTS = [0, 4, 8, 16, 32]
HDF5_FILE_PATH = "/home/nashah/scratch/data/pre_processed/preprocessed_straight_limbs_mod1_add_noise_0__include_weight_height_False__omit_contact_sobel_False__use_hover_False__mod_1__normalize_per_image_True.hdf5"
TRANSFORM = T.Normalize(
	mean=[26.201084, 11.778635, 11.731706],
	std	=[41.360558, 27.982226, 8.824089])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# Instantiate model
model = PressureNetMultiHead(in_channels=3, use_relu=True).to(device)
model.eval()

# 1) GPU Memory Benchmark
mem_results = []
if torch.cuda.is_available():
    for bs in BATCH_SIZES:
        try:
            torch.cuda.reset_peak_memory_stats(device)
            x = torch.randn(bs, 3, 128, 54, device=device)
            with torch.no_grad():
                _ = model(x)
            peak = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
            mem_results.append({'batch_size': bs, 'peak_mem_gb': peak})
        except RuntimeError as e:
            mem_results.append({'batch_size': bs, 'peak_mem_gb': None, 'error': str(e)})
else:
    print("CUDA not available: skipping GPU memory benchmark.")

df_mem = pd.DataFrame(mem_results)
print("=== GPU Memory Benchmark ===")
print(df_mem.to_string(index=False), "\n")

Using device: cuda

=== GPU Memory Benchmark ===
 batch_size  peak_mem_gb
        512     2.646523
       1024     3.989306
       2048     6.520613
       4096    11.741640 



In [21]:
# 2) GPU Throughput Benchmark (forward-only)
tp_results = []
if torch.cuda.is_available():
    for bs in BATCH_SIZES:
        row = df_mem[df_mem['batch_size'] == bs]
        if row['peak_mem_gb'].iloc[0] is None:
            continue
        x = torch.randn(bs, 3, 128, 54, device=device)
        torch.cuda.synchronize(device)
        t0 = time.time()
        with torch.no_grad():
            for _ in range(50):
                _ = model(x)
        torch.cuda.synchronize(device)
        t1 = time.time()
        tp_results.append({'batch_size': bs, 'iters_per_sec': 50 / (t1 - t0)})
else:
    print("CUDA not available: skipping GPU throughput benchmark.")

df_tp = pd.DataFrame(tp_results)
print("=== GPU Throughput Benchmark ===")
print(df_tp.to_string(index=False), "\n")

=== GPU Throughput Benchmark ===
 batch_size  iters_per_sec
        512      34.947134
       1024      18.052921
       2048       8.935462
       4096       4.452430 



In [23]:
# 3) DataLoader Speed Benchmark
dataset = HDF5Dataset(hdf5_file_path=HDF5_FILE_PATH, split='train', transform=TRANSFORM)
loader_results = []
bs = min(BATCH_SIZES)  # use smallest tested batch size for I/O test
for nw in WORKER_COUNTS:
    loader = DataLoader(dataset,
                        batch_size=bs,
                        shuffle=False,
                        num_workers=nw,
                        pin_memory=True,
                        # prefetch_factor=2,
                        # persistent_workers=True
                        )
    t0 = time.time()
    for i, (x, y) in enumerate(loader):
        if i >= 100:
            break
    t1 = time.time()
    loader_results.append({'num_workers': nw, 'time_100_batches_s': t1 - t0})
df_loader = pd.DataFrame(loader_results)
print("=== DataLoader Speed Benchmark ===")
print(df_loader.to_string(index=False))

=== DataLoader Speed Benchmark ===
 num_workers  time_100_batches_s
           0         2424.206598
           4          663.842162
           8          332.561983
          16          166.504899
          32          166.948061
